<a href="https://colab.research.google.com/github/Janeth172/EDP1/blob/main/CADENAS_ABSORBENTES_Janeth_CS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TAREA 5: CADENAS ABSORBENTES**

**EJERCICIO 1: LA RATA Y EL LABERINTO**

Se importan las librerías necesarias para realizar los cálculos y la simulación.

**SymPy** se utiliza para trabajar con matrices y fracciones exactas, mientras que **random** permite seleccionar aleatoriamente los movimientos de la rata.

La función **init_printing**() mejora la presentación de las matrices.

In [1]:
# Librerías
from numpy import *
import numpy as np
from sympy import *
import sympy as sp
import random as r

init_printing(use_unicode=True)

**Matriz de transición entre estados transitorios**

Esta matriz representa las probabilidades de pasar de un compartimento transitorio a otro. Solamente se consideran los estados del 0 al 6, porque el 7 y el 8 son *absorbentes*.

In [2]:
# Contruyendo matriz de transición
# Matriz con las probabilidades de movimiento entre los estados del 0 al 6

# S(1)/2, S(1)/3 y S(1)/4 para conservar las fracciones exactas
matriz_Q = Matrix([
    [0, S(1)/2, S(1)/2, 0, 0, 0, 0],
    [S(1)/3, 0, 0, S(1)/3, 0, 0, 0],
    [S(1)/3, 0, 0, S(1)/3, 0, 0, 0],
    [0, S(1)/4, S(1)/4, 0, S(1)/4, S(1)/4, 0],
    [0, 0, 0, S(1)/3, 0, 0, S(1)/3],
    [0, 0, 0, S(1)/3, 0, 0, S(1)/3],
    [0, 0, 0, 0, S(1)/2, S(1)/2, 0]
])

matriz_Q

⎡ 0   1/2  1/2   0    0    0    0 ⎤
⎢                                 ⎥
⎢1/3   0    0   1/3   0    0    0 ⎥
⎢                                 ⎥
⎢1/3   0    0   1/3   0    0    0 ⎥
⎢                                 ⎥
⎢ 0   1/4  1/4   0   1/4  1/4   0 ⎥
⎢                                 ⎥
⎢ 0    0    0   1/3   0    0   1/3⎥
⎢                                 ⎥
⎢ 0    0    0   1/3   0    0   1/3⎥
⎢                                 ⎥
⎣ 0    0    0    0   1/2  1/2   0 ⎦

**Matriz identidad menos Q que es \(I-Q\)**

Primero construir una matriz identidad de tamaño 7 × 7, ya que existen siete estados transitorios.

Después restar la matriz Q a la identidad. Para calcular la matriz fundamental de una cadena de Markov absorbente.

In [3]:
# Construyengo matriz I-Q para calcular la matriz fundamental
matriz_identidad = eye(7)

diferencia = matriz_identidad - matriz_Q

diferencia

⎡ 1    -1/2  -1/2   0     0     0     0  ⎤
⎢                                        ⎥
⎢-1/3   1     0    -1/3   0     0     0  ⎥
⎢                                        ⎥
⎢-1/3   0     1    -1/3   0     0     0  ⎥
⎢                                        ⎥
⎢ 0    -1/4  -1/4   1    -1/4  -1/4   0  ⎥
⎢                                        ⎥
⎢ 0     0     0    -1/3   1     0    -1/3⎥
⎢                                        ⎥
⎢ 0     0     0    -1/3   0     1    -1/3⎥
⎢                                        ⎥
⎣ 0     0     0     0    -1/2  -1/2   1  ⎦

**Matriz fundamental**

Esta matriz permite calcular el número esperado de visitas a los estados transitorios antes de que la rata llegue a un estado absorbente.

La necesitamos para obtener las probabilidades de absorción.

In [4]:
# Calcular la inversa de I-Q
matriz_fundamental = diferencia.inv()

matriz_fundamental

⎡7/4  9/8   9/8   1  3/8   3/8   1/4⎤
⎢                                   ⎥
⎢3/4  13/8  5/8   1  3/8   3/8   1/4⎥
⎢                                   ⎥
⎢3/4  5/8   13/8  1  3/8   3/8   1/4⎥
⎢                                   ⎥
⎢1/2  3/4   3/4   2  3/4   3/4   1/2⎥
⎢                                   ⎥
⎢1/4  3/8   3/8   1  13/8  5/8   3/4⎥
⎢                                   ⎥
⎢1/4  3/8   3/8   1  5/8   13/8  3/4⎥
⎢                                   ⎥
⎣1/4  3/8   3/8   1  9/8   9/8   7/4⎦

**Matriz de probabilidades hacia los estados absorbentes**

La matriz R contiene las probabilidades de pasar desde los estados transitorios hacia los estados absorbentes.

La primera columna corresponde al estado 7, donde se encuentra la comida, y la segunda al estado 8, donde ocurre la descarga.

In [5]:
# Probabilidades de pasar directamente a la comida o a la descarga
matriz_R = Matrix([
    [0, 0],
    [S(1)/3, 0],
    [0, S(1)/3],
    [0, 0],
    [S(1)/3, 0],
    [0, S(1)/3],
    [0, 0]
])

matriz_R

⎡ 0    0 ⎤
⎢        ⎥
⎢1/3   0 ⎥
⎢        ⎥
⎢ 0   1/3⎥
⎢        ⎥
⎢ 0    0 ⎥
⎢        ⎥
⎢1/3   0 ⎥
⎢        ⎥
⎢ 0   1/3⎥
⎢        ⎥
⎣ 0    0 ⎦

**Probabilidades analíticas**

La multiplicación de la matriz fundamental por la matriz R permite calcular la probabilidad de terminar en cada estado absorbente.
$$
B=FR
$$

In [6]:
# Multiplicar F por R para obtener las probabilidades de absorción
resultado_analitico = matriz_fundamental * matriz_R

# Resultados como porcentajes
porcentajes = (resultado_analitico * 100).evalf(4)

porcentajes

⎡50.0   50.0 ⎤
⎢            ⎥
⎢66.67  33.33⎥
⎢            ⎥
⎢33.33  66.67⎥
⎢            ⎥
⎢50.0   50.0 ⎥
⎢            ⎥
⎢66.67  33.33⎥
⎢            ⎥
⎢33.33  66.67⎥
⎢            ⎥
⎣50.0   50.0 ⎦

**Simulación del recorrido de la rata**

In [7]:
# Definimos una función del compartimento desde el cual comienza la rata
def recorrido_rata(estado_inicial):

    # Guardar las salidas disponibles desde cada compartimento
    conexiones = {
        0: [1, 2],
        1: [0, 3, 7],
        2: [0, 3, 8],
        3: [1, 2, 4, 5],
        4: [3, 6, 7],
        5: [3, 6, 8],
        6: [4, 5]
    }

    # La rata comienza en el compartimento indicado
    posicion = estado_inicial

    # Continuar hasta encontrar comida o recibir una descarga
    while posicion != 7 and posicion != 8:

        # Elegir una salida aleatoria entre las disponibles
        posicion = r.choice(conexiones[posicion])

    # Regresar 1 si encontró comida y 0 si recibió una descarga
    if posicion == 7:
        return 1
    else:
        return 0

**Probar un recorrido**

In [16]:
# Realizar una prueba comenzando en el estado 5
resultado = recorrido_rata(5)

print("Resultado del recorrido:", resultado)

Resultado del recorrido: 0


**Estimación mediante Montecarlo**

In [19]:
def estimar_probabilidad(inicio, repeticiones=100000):

    # Contar las veces que la rata encuentra comida
    llegadas_comida = 0

    # Repetir el recorrido el número de veces indicado
    for intento in range(repeticiones):

        resultado = recorrido_rata(inicio)

        # Sumar el resultado, que vale 1 si encontró comida
        llegadas_comida = llegadas_comida + resultado

    # Calcular la frecuencia relativa de éxito
    probabilidad = llegadas_comida / repeticiones

    return probabilidad

**Simulación desde todos los estados transitorios**

In [20]:
# Calcular la probabilidad de éxito desde cada compartimento
for estado in range(7):

    probabilidad = estimar_probabilidad(estado)

    print(
        f"Desde el estado {estado}: "
        f"Probablidad estimada es de {probabilidad * 100:.2f}% de llegar a la comida"
    )

Desde el estado 0: Probablidad estimada es de 49.78% de llegar a la comida
Desde el estado 1: Probablidad estimada es de 66.80% de llegar a la comida
Desde el estado 2: Probablidad estimada es de 33.34% de llegar a la comida
Desde el estado 3: Probablidad estimada es de 49.84% de llegar a la comida
Desde el estado 4: Probablidad estimada es de 66.56% de llegar a la comida
Desde el estado 5: Probablidad estimada es de 33.48% de llegar a la comida
Desde el estado 6: Probablidad estimada es de 50.15% de llegar a la comida


**EJERCICIO 2: SERPIENTES Y ESCALERAS**

Se importa la librería **random**, que permite generar resultados aleatorios. En este ejercicio la necesitamos para simular los lanzamientos del dado.

In [21]:
# Importando librerías
import random as r

**Definir el dado y el tablero**

In [22]:
# Posibles resultados al lanzar un dado de seis caras
caras_dado = [1, 2, 3, 4, 5, 6]

# Casillas que forman el tablero
tablero = list(range(1, 21))

**Función para simular una partida**

Se utiliza una función porque necesitamos repetir el mismo procedimiento muchas veces para calcular el promedio de tiradas.

La función devuelve el número total de lanzamientos necesarios para finalizar esa partida.

In [23]:
def simular_partida(casilla_inicial, dado, tablero):

    # Guardar las casillas que tienen serpientes o escaleras
    movimientos_especiales = {
        3: 11,   # Escalera
        13: 4,   # Serpiente
        15: 19,  # Escalera
        17: 10   # Serpiente
    }

    # Posición donde comienza el jugador
    posicion_actual = casilla_inicial

    # Contador de lanzamientos realizados
    tiradas = 0

    # Continuar jugando mientras no llegue a la meta
    while posicion_actual < tablero[-1]:

        # Obtener un número aleatorio entre 1 y 6
        resultado_dado = r.choice(dado)

        # Avanzar las casillas indicadas por el dado
        posicion_actual = posicion_actual + resultado_dado

        # Revisar si cayó en una serpiente o escalera
        if posicion_actual in movimientos_especiales:
            posicion_actual = movimientos_especiales[posicion_actual]

        # Contar la tirada realizada
        tiradas = tiradas + 1

    return tiradas

**Probar una partida**

In [26]:
# Simular una partida comenzando en la casilla 2
resultado = simular_partida(2, caras_dado, tablero)

print("Tiradas necesarias:", resultado)

Tiradas necesarias: 6


**Simulación de Montecarlo**

In [29]:
# Número de simulaciones
numero_partidas = 1000000

# Lista para guardar las tiradas de cada partida
resultados = []

# Repetir el juego desde la casilla 2
for i in range(numero_partidas):

    tiradas_partida = simular_partida(2, caras_dado, tablero)

    resultados.append(tiradas_partida)

**Calcular el promedio de tiradas**

$$
\overline{X}=\frac{X_1+X_2+\cdots+X_N}{N}
$$

In [30]:
# Sumar las tiradas y dividir entre el total de partidas
promedio = sum(resultados) / len(resultados)

print(f"Promedio de tiradas: {promedio:.4f}")

Promedio de tiradas: 6.6947
